In [2]:
from random import random, shuffle, choice, randint, seed, sample

N = 20
C = 5

seed(42)
# I handle the children as a tuple of tuples where the bools show if a child has a characteristics or not
children = tuple(
    tuple(choice([True, False]) for j in range(C))
    for i in range(N)
)

In [3]:
# The fitness is computed as the sum of differences of adjacent children
def fitness(children: tuple[tuple[bool]], sol: list[int]) -> int:
    fitness = 0

    # I overcomplicated the loop only to not use the list indeces which are inefficient
    prev_pos = sol[-1]
    for pos in sol:
        child = children[pos]
        prev_child = children[prev_pos]

        fitness += sum(1 if c1 != c2 else 0 for c1, c2 in zip(child, prev_child))
        
        prev_pos = pos

    return fitness

# To tweak the solutuion I swap two random children for a random amount of times.
def tweak(sol: list[int]) -> list[int]:
    n = len(sol)
    new_sol = sol[:]
    first = True

    # I noticed too many swaps tend to lower the fitness a lot, so I give low probability of additional ones
    while first or random() < 0.3:
        first = False
        pos1 = randint(0, n - 1)
        pos2 = randint(0, n - 1)

        aus = new_sol[pos1]
        new_sol[pos1] = new_sol[pos2]
        new_sol[pos2] = aus

    return new_sol

In [4]:
# For crossover I pick two consecutive elements of the first parent, find the positions of those elements in the second
# I bring those elements close and the elements in the middle are inverted
def cross_parents(p1: list[int], p2: list[int]) -> list[int]:
    pos = randint(0, len(p1) - 2)
    n1, n2 = p1[pos], p1[pos+1]

    pos1 = p2.index(n1)
    pos2 = p2.index(n2)
    pos1, pos2 = (pos1, pos2) if pos1 < pos2 else (pos2, pos1)

    return p2[:pos1] + p2[pos1:pos1+1] + p2[pos2:pos1:-1] + p2[pos2+1:]

# I create lambda children, each generated by 2 random parents
def create_children(population, la):
    children = []
    for _ in range(la):
        p1, p2 = tuple(sample(population, 2))

        children.append(cross_parents(p1, p2))

    return children


In [6]:
mu = 20
la = 20

population = [list(range(N)) for _ in range(mu)]
for sol in population:
    shuffle(sol)

iterations = 0
since_last_change = 0
max_fitness = -1
while since_last_change < 1000:
    new_solutions = create_children(population, la)
    new_solutions = [tweak(sol) for sol in new_solutions]

    population.extend(new_solutions)
    population.sort(key=lambda sol: fitness(children, sol), reverse=True)

    iterations += 1
    since_last_change += 1

    best_fitness = fitness(children, population[0])
    if best_fitness > max_fitness:
        since_last_change = 0
        max_fitness = best_fitness
        print(f"{iterations}) New Max fitness: {best_fitness}")

    population = population[:mu]

print(f"Best solution: {population[0]}")


1) New Max fitness: 56
3) New Max fitness: 62
4) New Max fitness: 66
12) New Max fitness: 68
24) New Max fitness: 70
31) New Max fitness: 72
59) New Max fitness: 74
66) New Max fitness: 76
90) New Max fitness: 78
124) New Max fitness: 80
Best solution: [14, 2, 10, 1, 17, 12, 9, 6, 18, 4, 8, 0, 15, 13, 7, 16, 3, 5, 11, 19]
